In [5]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("Device:", device)

Device: cuda


In [7]:
boston = fetch_openml(name="Boston", version=1, as_frame=False)
Xb = StandardScaler().fit_transform(boston["data"])
yb = MinMaxScaler((0, 1)).fit_transform(boston["target"].reshape(-1, 1)).astype(np.float32)
Xb = torch.tensor(Xb, dtype=torch.float32).to(device)
yb = torch.tensor(yb, dtype=torch.float32).to(device)

class NetReg(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, H),
            nn.ReLU(),
            nn.Linear(H, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

model = NetReg(Xb.shape[1], 64).to(device)
opt = optim.SGD(model.parameters(), lr=0.01)
mse = nn.MSELoss()

epochs = 200
t0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    opt.zero_grad()
    out = model(Xb)
    loss = mse(out, yb)
    loss.backward()
    opt.step()
    if epoch % 10 == 0 or epoch == 1 or epoch == epochs:
        print(f"[Boston][Epoch {epoch}/{epochs}] loss={loss.item():.6f} elapsed={time.perf_counter()-t0:0.1f}s")
print("Boston done.\n")

[Boston][Epoch 1/200] loss=0.049626 elapsed=0.0s
[Boston][Epoch 10/200] loss=0.047272 elapsed=0.0s
[Boston][Epoch 20/200] loss=0.044956 elapsed=0.1s
[Boston][Epoch 30/200] loss=0.042912 elapsed=0.1s
[Boston][Epoch 40/200] loss=0.041104 elapsed=0.1s
[Boston][Epoch 50/200] loss=0.039497 elapsed=0.2s
[Boston][Epoch 60/200] loss=0.038064 elapsed=0.2s
[Boston][Epoch 70/200] loss=0.036779 elapsed=0.2s
[Boston][Epoch 80/200] loss=0.035622 elapsed=0.3s
[Boston][Epoch 90/200] loss=0.034575 elapsed=0.3s
[Boston][Epoch 100/200] loss=0.033623 elapsed=0.4s
[Boston][Epoch 110/200] loss=0.032754 elapsed=0.4s
[Boston][Epoch 120/200] loss=0.031958 elapsed=0.4s
[Boston][Epoch 130/200] loss=0.031224 elapsed=0.4s
[Boston][Epoch 140/200] loss=0.030546 elapsed=0.5s
[Boston][Epoch 150/200] loss=0.029918 elapsed=0.5s
[Boston][Epoch 160/200] loss=0.029333 elapsed=0.5s
[Boston][Epoch 170/200] loss=0.028787 elapsed=0.5s
[Boston][Epoch 180/200] loss=0.028275 elapsed=0.5s
[Boston][Epoch 190/200] loss=0.027795 elap

In [8]:
mn = fetch_openml("mnist_784", version=1, as_frame=False)
X = mn["data"].astype(np.float32) / 255.0
y = mn["target"].astype(int)
X = StandardScaler().fit_transform(X)
X_t = torch.tensor(X, dtype=torch.float32).to(device)
y_t = torch.tensor(y, dtype=torch.long).to(device)

class NetMC(nn.Module):
    def __init__(self, D, H, C):
        super().__init__()
        self.fc1 = nn.Linear(D, H)
        self.fc2 = nn.Linear(H, C)
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h)

model2 = NetMC(X_t.shape[1], 128, 10).to(device)
opt2 = optim.SGD(model2.parameters(), lr=0.1)
crit = nn.CrossEntropyLoss()

epochs = 5
t0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    opt2.zero_grad()
    logits = model2(X_t)
    loss2 = crit(logits, y_t)
    loss2.backward(); opt2.step()
    preds = logits.argmax(dim=1)
    acc = (preds == y_t).float().mean().item()
    print(f"[MNIST][Epoch {epoch}/{epochs}] loss={loss2.item():.4f} acc={acc:.4f} elapsed={time.perf_counter()-t0:0.1f}s")
print("MNIST done.\n")

[MNIST][Epoch 1/5] loss=2.3315 acc=0.1086 elapsed=0.1s
[MNIST][Epoch 2/5] loss=2.2348 acc=0.2464 elapsed=0.1s
[MNIST][Epoch 3/5] loss=2.1439 acc=0.3807 elapsed=0.1s
[MNIST][Epoch 4/5] loss=2.0567 acc=0.4889 elapsed=0.1s
[MNIST][Epoch 5/5] loss=1.9721 acc=0.5510 elapsed=0.1s
MNIST done.



In [9]:
pima = fetch_openml("pima-indians-diabetes", version=1, as_frame=False)
Xp = StandardScaler().fit_transform(pima["data"]).astype(np.float32)
yp = pima["target"].astype(np.float32).reshape(-1, 1)

Xp_t = torch.tensor(Xp, dtype=torch.float32).to(device)
yp_t = torch.tensor(yp, dtype=torch.float32).to(device)

class NetBin(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

model3 = NetBin(Xp_t.shape[1], 32).to(device)
opt3 = optim.SGD(model3.parameters(), lr=0.01)
bcel = nn.BCELoss()

epochs = 300
t0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    opt3.zero_grad()
    out = model3(Xp_t)
    loss3 = bcel(out, yp_t)
    loss3.backward(); opt3.step()
    if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
        pred = (out > 0.5).float()
        acc3 = (pred == yp_t).float().mean().item()
        print(f"[Pima][Epoch {epoch}/{epochs}] loss={loss3.item():.6f} acc={acc3:.4f} elapsed={time.perf_counter()-t0:0.1f}s")
print("Pima done.")

[Pima][Epoch 1/300] loss=0.718411 acc=0.4010 elapsed=0.0s
[Pima][Epoch 50/300] loss=0.675064 acc=0.6354 elapsed=0.0s
[Pima][Epoch 100/300] loss=0.642996 acc=0.6602 elapsed=0.1s
[Pima][Epoch 150/300] loss=0.617600 acc=0.6797 elapsed=0.1s
[Pima][Epoch 200/300] loss=0.596439 acc=0.6966 elapsed=0.2s
[Pima][Epoch 250/300] loss=0.578312 acc=0.7227 elapsed=0.2s
[Pima][Epoch 300/300] loss=0.562635 acc=0.7357 elapsed=0.2s
Pima done.
